<a href="https://colab.research.google.com/github/jonathanr1212/wandb_take_home/blob/main/TSE_II_take_home_assignment_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Debug this Colab!

## Section 1

This colab represents a simple ML pipline, loading data, defining a model and fitting the model to the data. It has also been instrumented with Weights and Biases logging tools.

At Weights and Biases, we often help our users debug their pipelines -- both the ML code and the logging code from `wandb` integrated into it.

Your task is to debug this simple pipeline such that the model is able to learn and <u>perform reasonably well</u> on the given task, without changing the general structure of the model. As you do so, use comments and markdown cells to explain a bit about your process.


In [1]:
!pip install wandb

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms

import wandb

# Data Preprocessing

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

batch_size = 32

cifar10 = torchvision.datasets.CIFAR10(root='./data', download=True, transform=torchvision.transforms.ToTensor())
pivot = 40000
cifar10 = sorted(cifar10, key=lambda x: x[1])
train_set = torch.utils.data.Subset(cifar10, range(pivot))
val_set = torch.utils.data.Subset(cifar10, range(pivot, len(cifar10)))
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=True)

100%|██████████| 170M/170M [27:54<00:00, 102kB/s] 


In [73]:
class Network(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        ## first error has to do with MaxPooling2D. The correct method is nn.MaxPool2d according the pytorch docs https://docs.pytorch.org/docs/2.14/nn.html#pooling-layers
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        ## third error populates mat1 and mat2 shapes cannot be multiplied (32x400 and 600x120) error. This happened because the output size shrunk too much and output 400 instead of 600. Therefore, changing the Linear layer to 400 solves this.
        self.fc1 = nn.Linear(400, 120)
        self.fc2 = nn.Linear(120, 2)
        self.fc3 = nn.Linear(2, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        ##second error that populated was that torch.Flatten doesn't exist. According to the docs, it's torch.flatten https://docs.pytorch.org/docs/2.14/generated/torch.flatten.html
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = Network()

In [74]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0)

# Training and Validation

In this part, you will also need to additionally calculate training and validation accuracy and log it to Weights and Biases.

In [75]:
with wandb.init(project = 'Tier-1-Test', save_code=True) as run:
    for epoch in range(5):
        current_loss = 0

        model.train()

        for i, data in enumerate(train_loader):
            images, labels = data

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            current_loss += loss

        run.log({'train_loss' : current_loss / (i + 1)})

        model.eval()

        current_loss = 0

        for i, data in enumerate(val_loader):
            images, labels = data
            outputs = model(images)

            loss = criterion(outputs, labels)

            current_loss += loss

        run.log({'val_loss' : current_loss / (i + 1)})

train_loss,█▁▃▃▃
val_loss,▁▃▅▆█
train_loss,2.10758
val_loss,107.90445


Now that you have completed the task, please write 3-5 lines sharing your approach to the problem and how you went about solving this task.

```
I went through a very step by step way of trying to solve the first problem where I ran the cell and saw the error produced. It got roughly 3 errors before the notebook was able to run properly. From there, I can see that the training loss and validation loss was going up when the loss should be going down. I tried to work with the optimizer given (SGD instead of Adam) but while lower the learning rate and decreasing the momentum helped with getting the training loss to decrease, it did not help on the validation set. I tried also adding a weight_decay to no avail.
```

## Section 2

For this section, your task is to write code which can solve the following problem. We have provided a few unit tests to aid you with your task, but these are not comprehensive and you *should* write a few tests of your own to make sure your task runs perfectly.


```
Write a program to calculate the nth root of a given number x. You will be given the variables x and n. Your root should be close to the true value to atleast 2 decimal points of precision.

Constraints:
- You are NOT allowed to use exp(), pow(), the exponentiation operator (**) or any pre-built exponentiation methods.
- x is guaranteed to be a positive real number.
- n is guaranteed to be a positive integer.
```

In [ ]:
def root(x : float, n : int):
  # YOUR CODE HERE
  ...

### Unit Tests

(You are allowed to use pre built exponentiation methods to test against)

In [ ]:
THRESHOLD = 1e-2

def is_approximately_equal(a, b):
  return abs(a - b) <= THRESHOLD

In [ ]:
assert is_approximately_equal(root(100, 2), 100 ** 0.5)
assert is_approximately_equal(root(50, 2), 50 ** 0.5)

assert is_approximately_equal(root(30, 5), 30 ** 0.2)
assert is_approximately_equal(root(1, 2), 1)